In [13]:
import pandas as pd
import numpy as np

In [14]:
live_avg_mega = pd.read_csv("../samples/mega_processed/live_avg.csv")
live_avg_ollama = pd.read_csv("../samples/live_avg.csv")

In [15]:
def weights_builder(df):
    gpu_util = df['GPU Utilization (%)']
    mem_utill = df['Memory Utilization (%)']
    rt = df["Resonse Time"]
    rt_avg = np.average(rt)
    gpu_std = np.std(gpu_util)
    mem_std = np.std(mem_utill)
    w1 = 1/gpu_std
    w2 = 1/mem_std
    w3 = -np.exp(rt_avg/(25/np.log(2)))
    weights = [w1, w2, w3]
    weights_norm = weights / np.sum(np.abs(weights)) 
    return weights_norm


In [16]:
live_v = pd.read_csv("../samples/trial/live_metrics.csv")
live_a = pd.read_csv("../samples/mega/live_metrics.csv")

In [17]:
verbose_v = pd.read_csv("../samples/trial/verbose_statements.csv")
verbose_a = pd.read_csv("../samples/mega/verbose_statements.csv")

In [18]:
live_v = live_v.dropna()
live_a = live_a.dropna()
verbose_v = verbose_v.dropna()
verbose_a = verbose_a.dropna()

In [19]:
weights_a = weights_builder(live_avg_mega)
weights_v = weights_builder(live_avg_ollama)

In [20]:
weights_a, weights_v

(array([ 0.32220293,  0.61772018, -0.0600769 ]),
 array([ 0.36548244,  0.53310189, -0.10141568]))

In [21]:
avg_v_gpu = np.average(live_v['GPU Utilization (%)'])
avg_v_mem = np.average(live_v['Memory Utilization (%)'])
avg_a_gpu = np.average(live_a['GPU Utilization (%)'])
avg_a_mem = np.average(live_a['Memory Utilization (%)'])
avg_v_rt = np.average(verbose_v['total_duration'])
avg_a_rt = np.average(verbose_a['total_duration'])

In [22]:
averages_a = [avg_a_gpu, avg_a_mem, avg_a_rt]
averages_v = [avg_v_gpu, avg_v_mem, avg_v_rt]

In [23]:
score_v = np.dot(weights_v, averages_v)
score_a = np.dot(weights_a, averages_a)

In [24]:
score_a, score_v

(0.10258279941504034, -0.23276617703636576)